# 04 - Difference-in-differences

Difference-in-differences is useful when treatment starts at a specific time for one group and we can compare the change against a control group.


## Causal question
State the causal question this notebook answers before reading numeric output.

## Causal setup
- Treatment: define the treatment variable and intervention of interest.
- Outcome: define the outcome variable being affected by treatment.
- Covariates: list observed confounders included in the design/diagnostics.
- Unit of analysis: specify the observational unit used in this notebook.

## Estimand
Specify the target estimand (ATE, ATT, CATE, etc.) and how it maps to model coefficients.

## Identification assumptions
Enumerate the identification assumptions required for a causal interpretation (for example ignorability, overlap, no interference).

## Uncertainty and limitations
Report interval estimates, sensitivity checks, and at least one limitation of the design.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step execution
This cell performs the next computation. Read the result and tie it back to the causal setup before moving on.


In [ ]:
import statsmodels.formula.api as smf

from causal_inference_lab.data_generators import make_did_panel
from causal_inference_lab.plotting import plot_did_trends

dataset = make_did_panel(n_units=600, n_periods=8, seed=7)
data = dataset.data

data.head()


## Visual trend check

The key assumption is parallel trends: without treatment, the treated and control groups would have followed similar trends.


In [ ]:
fig = plot_did_trends(data)
plt.show()


## Two-way fixed effects-style regression

We estimate the treatment effect using unit and time fixed effects.


In [ ]:
model = smf.ols("outcome ~ treatment + C(unit) + C(time)", data=data).fit(
    cov_type="cluster",
    cov_kwds={"groups": data["unit"]},
)

estimate = model.params["treatment"]
print(f"Estimated DiD effect: {estimate:.3f}")
print(f"True effect:          {dataset.true_ate:.3f}")


## Placebo pre-period check

In a real project, we would test for differential pre-trends. Here we create a placebo post indicator inside the pre-treatment period.


In [ ]:
pre_data = data[data["time"] < 4].copy()
pre_data["placebo_post"] = (pre_data["time"] >= 2).astype(int)
pre_data["placebo_treatment"] = pre_data["treated_group"] * pre_data["placebo_post"]

placebo_model = smf.ols(
    "outcome ~ placebo_treatment + C(unit) + C(time)",
    data=pre_data,
).fit(cov_type="cluster", cov_kwds={"groups": pre_data["unit"]})

print(f"Placebo estimate: {placebo_model.params['placebo_treatment']:.3f}")


**Interpretation.** A small placebo estimate supports the design, but it does not prove parallel trends. It only provides evidence that the pre-period does not show a large differential shift.
